# Simple Fraud Agent (E2B Full DB)

This notebook creates a minimal fraud agent that only uses the E2B template database at `/data/data.db`.
No CSV scoping is used.

In [1]:
%pip install -q aieng-agents sqlglot python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from aieng.agents.tools import CodeInterpreter
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import json
import sqlglot
from aieng.agents import set_up_logging, setup_langfuse_tracer

load_dotenv()
set_up_logging()

# This is the name you will see on traces
EXPERIMENT_TRACE_NAME = "fraud-agent-simple-e2b-march25"
tracer = setup_langfuse_tracer(service_name=EXPERIMENT_TRACE_NAME)

print(f"Langfuse tracing enabled: {EXPERIMENT_TRACE_NAME}")

# Keep this model simple and deterministic for the notebook workflow.
AGENT_LLM_NAME = "gemini-3-pro-preview" #"gemini-2.5-pro"
E2B_SQLITE_PATH = "/data/data.db"

# Replace template_name if your environment uses a different E2B template.
shared_code_interpreter = CodeInterpreter(
    template_name="lobsuu8phhb16r4r04yr"
)

print("Simple E2B fraud agent config ready.")

Langfuse tracing enabled: fraud-agent-simple-e2b-march25
Simple E2B fraud agent config ready.


In [6]:
_ALLOWED_ROOTS = {"select", "union", "paren"}
_FORBIDDEN_NODES = {
    "create",
    "insert",
    "update",
    "delete",
    "drop",
    "alter",
    "truncate_table",
    "merge",
    "command",
    "pragma",
    "attach",
    "detach",
    "set",
}
_FORBIDDEN_TABLES = {"fraud_labels"}
_FORBIDDEN_SQL_FUNCTIONS = {
    "percentile",
    "percentile_cont",
    "percentile_disc",
    "approx_percentile",
}


def _normalize_code_interpreter_result(raw_result):
    if isinstance(raw_result, str):
        return json.loads(raw_result)
    if isinstance(raw_result, dict):
        return raw_result
    if hasattr(raw_result, "model_dump"):
        return raw_result.model_dump()
    raise TypeError(f"Unsupported E2B result type: {type(raw_result)!r}")


def _extract_stdout_text(raw_result) -> str:
    payload = _normalize_code_interpreter_result(raw_result)
    error = payload.get("error")
    if error:
        raise RuntimeError(f"E2B execution failed: {error}")

    stdout = payload.get("stdout", [])
    if isinstance(stdout, list):
        return "\n".join(str(line) for line in stdout)
    return str(stdout)


def _validate_read_only_sql(query: str) -> str:
    cleaned_query = query.strip().rstrip(";")
    if not cleaned_query:
        raise ValueError("SQL query must not be empty.")

    expressions = sqlglot.parse(cleaned_query, read="sqlite")
    if len(expressions) != 1:
        raise ValueError("Only a single SQL statement is allowed.")

    expression = expressions[0]
    if expression.key.lower() not in _ALLOWED_ROOTS:
        raise ValueError("Only read-only SELECT queries are allowed.")

    for node in expression.walk():
        if node.key.lower() in _FORBIDDEN_NODES:
            raise ValueError(f"Forbidden SQL operation detected: {node.key}")
        if node.key.lower() == "anonymous":
            function_name = node.name.lower() if node.name else ""
            if function_name in _FORBIDDEN_SQL_FUNCTIONS:
                raise ValueError(f"Unsupported SQLite function detected: {function_name}")

        if node.key.lower() == "table":
            table_name = node.name.lower() if node.name else ""
            if table_name in _FORBIDDEN_TABLES:
                raise ValueError(f"Queries against '{table_name}' are not allowed.")

    return cleaned_query


async def get_database_schema() -> str:
    code = f"""
import sqlite3

conn = sqlite3.connect({E2B_SQLITE_PATH!r})
rows = conn.execute(
    '''
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
      AND name NOT LIKE 'sqlite_%'
      AND name != 'fraud_labels'
    ORDER BY type, name
    '''
).fetchall()

sections = []
for name, relation_type in rows:
    table_info = conn.execute(f'PRAGMA table_info("{{name}}")').fetchall()
    columns = ', '.join(f"{{column[1]}} {{column[2] or 'TEXT'}}" for column in table_info)
    sections.append(f"{{relation_type}}: {{name}}\\n  columns: {{columns}}")

print('\\n\\n'.join(sections) or 'No tables found.')
conn.close()
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _extract_stdout_text(raw_result)


async def run_readonly_query(query: str, row_limit: int = 200) -> dict:
    safe_query = _validate_read_only_sql(query)

    code = f"""
import json
import sqlite3

query = {safe_query!r}
conn = sqlite3.connect({E2B_SQLITE_PATH!r})
conn.row_factory = sqlite3.Row
cursor = conn.execute(query)
columns = [c[0] for c in (cursor.description or [])]
rows = [dict(r) for r in cursor.fetchmany({int(row_limit)})]
payload = {{
    'query': query,
    'columns': columns,
    'rows': rows,
    'row_limit': {int(row_limit)},
}}
print(json.dumps(payload, default=str))
conn.close()
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return json.loads(_extract_stdout_text(raw_result).strip())


async def detect_high_amount_transactions(min_amount: float = 5000, limit: int = 20) -> dict:
    query = f"""
    SELECT transaction_id, date, client_id, merchant_id, merchant_state, amount, mcc
    FROM transactions
    WHERE CAST(amount AS REAL) >= {float(min_amount)}
    ORDER BY CAST(amount AS REAL) DESC
    LIMIT {int(limit)}
    """
    result = await run_readonly_query(query, row_limit=limit)
    return {
        "min_amount": float(min_amount),
        "count": len(result.get("rows", [])),
        **result,
    }


async def detect_high_velocity_clients(min_txns: int = 5) -> dict:
    query = f"""
    SELECT
        client_id,
        COUNT(*) AS transaction_count,
        SUM(CAST(amount AS REAL)) AS total_amount,
        AVG(CAST(amount AS REAL)) AS avg_amount
    FROM transactions
    GROUP BY client_id
    HAVING COUNT(*) >= {int(min_txns)}
    ORDER BY transaction_count DESC, total_amount DESC
    LIMIT 50
    """
    result = await run_readonly_query(query, row_limit=50)
    return {
        "min_txns": int(min_txns),
        "count": len(result.get("rows", [])),
        **result,
    }


async def detect_risky_merchants(min_txns: int = 10) -> dict:
    query = f"""
    SELECT
        merchant_id,
        COUNT(*) AS transaction_count,
        COUNT(DISTINCT client_id) AS unique_clients,
        COUNT(DISTINCT merchant_state) AS unique_states,
        SUM(CAST(amount AS REAL)) AS total_amount
    FROM transactions
    GROUP BY merchant_id
    HAVING COUNT(*) >= {int(min_txns)}
    ORDER BY transaction_count DESC, unique_states DESC
    LIMIT 50
    """
    result = await run_readonly_query(query, row_limit=50)
    return {
        "min_txns": int(min_txns),
        "count": len(result.get("rows", [])),
        **result,
    }

print("Read-only SQL + simple fraud tools ready.")

Read-only SQL + simple fraud tools ready.


In [ ]:
SIMPLE_FRAUD_AGENT_INSTRUCTIONS = f"""
You are a fraud detection analyst working on SQLite DB: {E2B_SQLITE_PATH}

Workflow:
1) Start by calling get_database_schema.
2) Use tool calls for evidence-based analysis.
3) Use only read-only SQL tools.
4) Do not use PERCENTILE/PERCENTILE_CONT/PERCENTILE_DISC. SQLite in this environment does not support them.
5) CRITICAL: Always include at least 3 sample transaction IDs in your findings. For each finding, cite specific transaction_id values from the database.

SQL Safety Rules (strict):
1) Generate exactly one SQL statement per tool call.
2) Allowed query shapes:
   - SELECT ... FROM ...
   - WITH ... SELECT ...
3) Forbidden operations/keywords:
   INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, REPLACE, TRUNCATE,
   PRAGMA, ATTACH, DETACH, VACUUM, BEGIN, COMMIT, ROLLBACK.
4) Return plain SQL only for SQL tool calls:
   - no markdown fences
   - no explanations
   - no comments in SQL
5) Prefer explicit column names and include LIMIT for exploratory queries.

Tools:
- detect_high_amount_transactions
- detect_high_velocity_clients
- detect_risky_merchants
- run_readonly_query

Output Format:
ANALYSIS:
<short summary>

FINDINGS:
- <key finding with supporting metrics and at least 3 sample transaction IDs (e.g., 'transaction_id: 12345, 12346, 12347')>

RISK:
- <risk level and why>

RECOMMENDATIONS:
- <actionable next step>
""".strip()

In [ ]:
simple_fraud_agent = Agent(
    name="simple_fraud_agent",
    model=AGENT_LLM_NAME,
    instruction=SIMPLE_FRAUD_AGENT_INSTRUCTIONS,
    tools=[
        get_database_schema,
        detect_high_amount_transactions,
        detect_high_velocity_clients,
        detect_risky_merchants,
        run_readonly_query,
    ],
)

simple_session_service = InMemorySessionService()
simple_runner = Runner(
    app_name=simple_fraud_agent.name,
    agent=simple_fraud_agent,
    session_service=simple_session_service,
)

simple_session = await simple_session_service.create_session(
    app_name=simple_fraud_agent.name,
    user_id="notebook-user",
    state={},
)


async def ask_simple_fraud_agent(question: str, fresh_session: bool = True) -> str:
    session_id = simple_session.id
    if fresh_session:
        new_session = await simple_session_service.create_session(
            app_name=simple_fraud_agent.name,
            user_id="notebook-user",
            state={},
        )
        session_id = new_session.id

    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_chunks = []

    async for event in simple_runner.run_async(
        user_id="notebook-user",
        session_id=session_id,
        new_message=content,
    ):
        if not event.is_final_response() or not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                final_chunks.append(part.text)

    return "\n".join(chunk for chunk in final_chunks if chunk).strip()


print("Simple fraud agent ready.")
print("Use: await ask_simple_fraud_agent('Find suspicious patterns in the database')")

Simple fraud agent ready.
Use: await ask_simple_fraud_agent('Find suspicious patterns in the database')


In [5]:
response = await ask_simple_fraud_agent(
    "Identify the most suspicious activity in the full database and list concrete evidence."
)
print(response)

/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/adk/runners.py:1481: DeprecationWarning: deprecated
  save_input_blobs_as_artifacts=run_config.save_input_blobs_as_artifacts,
/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/genai/_api_client.py:755: DeprecationWarning: Inheritance class AiohttpClientSession from ClientSession is discouraged
  class AiohttpClientSession(aiohttp.ClientSession):  # type: ignore[misc]


14:18:49.378 invocation
14:18:49.381   invoke_agent simple_fraud_agent
14:18:49.385     call_llm


14:18:53.261       execute_tool get_database_schema


14:18:53.893     call_llm
14:18:58.031       execute_tool detect_high_velocity_clients


14:19:01.692     call_llm
14:19:06.725       execute_tool run_readonly_query


14:19:07.265     call_llm
14:19:22.557       execute_tool run_readonly_query


14:19:26.832     call_llm
14:19:34.192       execute_tool detect_risky_merchants


14:19:38.404     call_llm
14:19:44.092       execute_tool run_readonly_query


14:19:45.673     call_llm
14:19:54.371       execute_tool run_readonly_query


14:19:54.915     call_llm
ANALYSIS:
The investigation focused on identifying suspicious activity within the database. The analysis began by identifying clients with an unusually high volume of transactions. This led to the discovery of a single client (ID 909) with over 29,000 transactions. Further analysis revealed a strong link between this client and a specific merchant (ID 59935), where a significant portion of these transactions occurred. The pattern of activity, involving multiple cards from a single client at a single merchant with a high number of transactions, is a strong indicator of a card testing attack.

FINDINGS:
- **High-Velocity Client:** Client ID 909 conducted an anomalously high number of transactions (29,171).
- **High-Transaction Merchant:** Merchant ID 59935 was associated with an extremely high volume of transactions (127,218).
- **Client-Merchant Link:** A substantial number of transactions (9,496) were identified between client ID 909 and merchant ID 59935.
- *